# Notebook 31 — Multi-Model Routing

Route LLM calls to the right model automatically based on cost, quality, latency, load, or content rules — with health tracking and fallback chains.

| Class | Strategy |
|---|---|
| `CostRouter` | cheapest model for the token budget |
| `QualityRouter` | highest quality_score |
| `LatencyRouter` | lowest expected latency |
| `RoundRobinRouter` | even distribution |
| `ContentRouter` | rule-based on input content |
| `AdaptiveRouter` | epsilon-greedy, learns from outcomes |
| `FallbackRouter` | try in order, move to next on error |
| `ModelRouter` | unified facade wrapping any strategy |

## 1. Define a model pool

In [ ]:
from multigen.routing import (
    ModelSpec, ModelPool, ModelCapability,
    ModelRouter, CostRouter, QualityRouter, LatencyRouter,
    RoundRobinRouter, ContentRouter, AdaptiveRouter, FallbackRouter,
    RoutingDecision,
)

pool = ModelPool([
    ModelSpec(
        name='gpt-4o',
        provider='openai',
        cost_per_1k_tokens=0.005,
        quality_score=0.95,
        expected_latency_ms=800,
        capabilities=[ModelCapability.TEXT, ModelCapability.CODE, ModelCapability.VISION],
    ),
    ModelSpec(
        name='gpt-3.5-turbo',
        provider='openai',
        cost_per_1k_tokens=0.001,
        quality_score=0.75,
        expected_latency_ms=300,
    ),
    ModelSpec(
        name='claude-3-haiku',
        provider='anthropic',
        cost_per_1k_tokens=0.00025,
        quality_score=0.80,
        expected_latency_ms=250,
    ),
])

print('Pool available:', [m.name for m in pool.available()])

## 2. Cost, Quality, Latency routers

In [ ]:
async def show_route(router_name, router):
    decision = await router.route({'_token_estimate': 1000})
    print(f'{router_name:<20} → {decision.model.name:<20} ({decision.reason})')

cost_router    = ModelRouter(pool=pool, strategy=CostRouter())
quality_router = ModelRouter(pool=pool, strategy=QualityRouter())
latency_router = ModelRouter(pool=pool, strategy=LatencyRouter())
rr_router      = ModelRouter(pool=pool, strategy=RoundRobinRouter())

await show_route('CostRouter',       cost_router)
await show_route('QualityRouter',    quality_router)
await show_route('LatencyRouter',    latency_router)
await show_route('RoundRobin[1]',    rr_router)
await show_route('RoundRobin[2]',    rr_router)  # cycles

## 3. Content-based routing

In [ ]:
# Route code questions to GPT-4o, everything else to Haiku
content = ContentRouter()
content.add_rule(lambda ctx: 'code' in str(ctx.get('prompt','')).lower(), 'gpt-4o')
content.add_rule(lambda ctx: 'vision' in str(ctx.get('prompt','')).lower(), 'gpt-4o')
content.set_default('claude-3-haiku')

cr = ModelRouter(pool=pool, strategy=content)

for prompt in ['Write Python code to sort a list', 'Explain quantum entanglement', 'Describe this image']:
    d = await cr.route({'prompt': prompt})
    print(f'{prompt[:40]:<42} → {d.model.name}')

## 4. Adaptive router (epsilon-greedy)

In [ ]:
adaptive = AdaptiveRouter(epsilon=0.1)
ar = ModelRouter(pool=pool, strategy=adaptive)

# Simulate many calls — record that GPT-4o gets high scores
for _ in range(20):
    d = await ar.route({})
    # Simulate: gpt-4o gets 0.9, others 0.6
    score = 0.9 if d.model.name == 'gpt-4o' else 0.6
    adaptive.record_outcome(d.model.name, success=(score > 0.7))

stats = adaptive.stats()
for name, s in stats.items():
    print(f'{name}: success_rate={s["success_rate"]:.2f} calls={s["calls"]}')

## 5. Fallback router

In [ ]:
async def call_fn(model):
    if model.name == 'gpt-4o':
        raise ConnectionError('Primary down')
    return f'Response from {model.name}'

fb = FallbackRouter(pool=pool, strategy=QualityRouter())
result, error = await fb.route_with_fallback(call_fn)
print(f'Got: {result}  (primary error: {error})')

## 6. Health tracking

In [ ]:
# Simulate failures
for _ in range(3):
    pool.mark_failure('gpt-4o', 'RateLimitError')
pool.mark_success('gpt-3.5-turbo')
pool.mark_success('claude-3-haiku')

report = pool.health_report()
for name, h in report.items():
    print(f'{name}: healthy={h["is_healthy"]} errors={h["error_count"]} success_rate={h["success_rate"]:.2f}')

## 7. Router stats

In [ ]:
print('cost_router stats:', cost_router.stats())
print('adaptive stats:   ', adaptive.stats())